# DSpark: Confidence-Scheduled Speculative Decoding

**Goal**: After this session, you will be able to implement DSpark's two contributions from memory — a *semi-autoregressive* draft head (a low-rank Markov transition bias added to a parallel backbone's base logits) and *confidence-scheduled verification* (a per-position acceptance predictor feeding a throughput-maximizing prefix scheduler).
**Time**: 60 minutes (a hard ceiling — most parts are short recall; the scheduler is the one meaty part).

## What and Why
Speculative decoding speeds up LLM inference: a cheap **draft model** proposes a block of γ tokens, the **target model** verifies them in one parallel pass, and the longest correct prefix is accepted. *Parallel* drafters propose all γ tokens at once (fast) but score each position independently, so the draft suffers rapid **suffix decay** ("of course" / "no problem" → "of problem"). DSpark keeps the parallel backbone but bolts on a **lightweight sequential head** that injects intra-block dependency, then **schedules** how much of each request's draft to actually verify based on predicted acceptance and live engine load.

## Key Formulas
Semi-autoregressive block factorization (sequential head adds a transition bias to the base logits):
$$p_k(v \mid x_0, x_{<k}) = \mathrm{softmax}\big(U_k(v) + B_k(x_0, x_{<k}, v)\big), \qquad B(x_{k-1},\cdot) = W_1[x_{k-1}]\,W_2 \in \mathbb{R}^V$$

Analytical per-step acceptance rate (the confidence head's supervision target):
$$c_k^{*} = 1 - \tfrac{1}{2}\,\lVert p_k^{d} - p_k^{t} \rVert_1 \;=\; \sum_v \min\!\big(p_k^d(v),\, p_k^t(v)\big)$$

Hardware-aware throughput to maximize, with prefix survival $a_{r,j}=\prod_{i\le j} c_{r,i}$:
$$\Theta = \tau \cdot \mathrm{SPS}(B), \quad B = \sum_r (1+\ell_r), \quad \tau = \sum_r\Big(1 + \sum_{j=1}^{\ell_r} a_{r,j}\Big)$$

**Where:**
- $U_k$ — base logit vector at draft position $k$ from the frozen parallel backbone (shape $V$).
- $B_k$ — prefix-dependent transition bias added to $U_k$; the Markov head makes it depend only on the previous token $x_{k-1}$.
- $W_1 \in \mathbb{R}^{V\times r}$ — low-rank Markov embedding lookup (shared with the confidence head); $W_2 \in \mathbb{R}^{r\times V}$ — projection back to vocab.
- $x_0$ — the anchor / bonus token from the previous verification round.
- $p_k^d, p_k^t$ — draft and target distributions at position $k$; $\lVert\cdot\rVert_1$ is total variation $\times 2$.
- $c_k^{*}$ — analytical probability that draft token $k$ survives target verification given the prefix is accepted.
- $a_{r,j}$ — survival prob. of request $r$'s length-$j$ prefix; $\ell_r$ — verification length chosen for request $r$; $\mathrm{SPS}(B)$ — profiled engine throughput (forward-pass steps/sec) at batch size $B$ tokens.

## Component Breakdown
- **Part 1 — Markov head**: low-rank transition bias $B(x_{k-1},\cdot)=W_1[x_{k-1}]W_2$ (a $V\times V$ matrix stored as rank-$r$ factors).
- **Part 2 — Semi-autoregressive sampling**: draft the block left-to-right, each position conditioning on the token *actually sampled* at the previous step.
- **Part 3 — Confidence head**: $c_k = \sigma\!\big(w^\top[h_k; W_1[x_{k-1}]]\big)$, a scalar acceptance predictor from backbone hidden + Markov embedding.
- **Part 4 — Acceptance label**: the analytical target $c_k^{*}=1-\tfrac12\lVert p^d-p^t\rVert_1$ that supervises Part 3.
- **Part 5 — Prefix scheduler (Algorithm 1)**: greedily admit draft tokens by cumulative survival to maximize $\Theta=\tau\cdot\mathrm{SPS}(B)$, early-stopping for losslessness.
- **Part 6 — Combined training loss**: position-weighted $\mathcal{L}=\alpha_{ce}\mathcal{L}_{ce}+\alpha_{tv}\mathcal{L}_{tv}+\alpha_{conf}\mathcal{L}_{conf}$.

## Reference
DSpark: Confidence-Scheduled Speculative Decoding with Semi-Autoregressive Generation (DeepSeek-AI & Peking University). Code: https://github.com/deepseek-ai/DeepSpec


In [ ]:
import itertools
import torch
import torch.nn as nn
import torch.nn.functional as F
from jaxtyping import Float, Int
from torch import Tensor

torch.manual_seed(0)

# ---- DSpark dimensions (scaled down from the paper for a CPU-only, self-contained run) ----
V     = 256   # vocabulary size    (paper: 150K+; shrunk so a full V×V transition matrix is inspectable)
r     = 32    # Markov head rank   (paper: 256)
d     = 64    # backbone hidden dim
GAMMA = 7     # draft block size γ  (paper: 7)

# ---- Frozen "parallel backbone" (DFlash) outputs for one draft block ----------------------
# In real DSpark these come from ONE forward pass over the block [anchor, mask, mask, ...].
# Position k yields a base logit U_k and a hidden state h_k.
base_logits: Float[Tensor, "gamma V"] = torch.randn(GAMMA, V)  # U_1..U_γ  (== backbone logits)
hidden:      Float[Tensor, "gamma d"] = torch.randn(GAMMA, d)  # h_1..h_γ  (== backbone hidden_states)
anchor_token: int = 17                                         # x_0, the bonus token from the previous round

# ---- Shared Markov embedding W_1 (used by BOTH the Markov head AND the confidence head) ----
W1 = nn.Embedding(V, r)           # W_1 ∈ R^{V×r}: rank-r embedding of the previously sampled token
W2 = nn.Linear(r, V, bias=False)  # W_2 ∈ R^{r×V}: projection of that embedding back to a vocab-size bias
conf_proj = nn.Linear(d + r, 1)   # confidence head w^T over [h_k ; W_1[x_{k-1}]]

# ---- Profiled engine throughput: forward-pass steps/sec at batch size B tokens -------------
# Monotonically decreasing — larger verification batches run slower per step (the system load curve).
def sps(B: float) -> float:
    return 100.0 / (B + 20.0)

print(f"V={V}  r={r}  d={d}  GAMMA={GAMMA}")
print(f"base_logits {tuple(base_logits.shape)}   hidden {tuple(hidden.shape)}   anchor={anchor_token}")


## Part 1: Markov head — the low-rank transition bias

A parallel backbone gives a base logit $U_k$ at every position but cannot make position $k$ depend on what was sampled at $k-1$. The Markov head fixes this with a **first-order transition bias** $B(x_{k-1},\cdot)$ added to $U_k$. In principle that bias is a full $V\times V$ matrix (one bias vector per possible previous token); DSpark stores it as the rank-$r$ factors $W_1\in\mathbb{R}^{V\times r}$, $W_2\in\mathbb{R}^{r\times V}$ — the **same low-rank trick you used for LoRA**. Given the previous token, the bias is $B(x_{k-1},\cdot)=W_1[x_{k-1}]\,W_2$.

**Predict before you code**: for a single previous token id, what shape is the returned bias vector? If you stacked the bias for *every* possible previous token, what shape would that matrix be, and what is its rank at most?

In [ ]:
# commit your predictions before coding:
predicted_bias_shape = (...,)       # shape of markov_bias(one_token_id)
predicted_full_rank  = ...          # max rank of the full V×V transition matrix W1@W2

def markov_bias(prev_token: Int[Tensor, "..."], W1: nn.Embedding, W2: nn.Linear) -> Float[Tensor, "... V"]:
    """Transition bias B(x_{k-1}, ·) = W_1[x_{k-1}] W_2  ∈ R^V."""
    # Implement from memory (low-rank: lookup then project)
    pass

In [ ]:
# --- Part 1 Validation ---
torch.manual_seed(42)
tok = torch.tensor(5)
bias = markov_bias(tok, W1, W2)

print(f"  You predicted bias shape {predicted_bias_shape}; actual {tuple(bias.shape)}")
print(f"  You predicted full-matrix rank {predicted_full_rank}; actual cannot exceed r={r}")
assert bias.shape == (V,), f"Shape: expected ({V},), got {tuple(bias.shape)}"
print(f"  Shape: {tuple(bias.shape)} -- correct")
print(f"  Range: [{bias.min():.4f}, {bias.max():.4f}]   Mean: {bias.mean():.4f}")

# Reference: the bias for token t must equal row t of the full low-rank transition matrix W1 @ W2.
B_full = W1.weight @ W2.weight.t()          # (V, V), the implicit transition matrix
ref = B_full[tok]
max_diff = (bias - ref).abs().max()
try:
    assert torch.allclose(bias, ref, atol=1e-5), f"Max diff {max_diff:.2e}"
except AssertionError:
    print(f"  YOUR:     {bias[:6].tolist()}")
    print(f"  EXPECTED: {ref[:6].tolist()}")
    raise
print(f"  Reference match (row of W1@W2) -- correct (max diff {max_diff:.2e})")

# Batched call must also work (one bias per position in a block).
batch_bias = markov_bias(torch.arange(GAMMA), W1, W2)
assert batch_bias.shape == (GAMMA, V), f"Batched shape: {tuple(batch_bias.shape)}"
print(f"  Batched {tuple(batch_bias.shape)} -- correct\nPart 1 complete.")

## Part 2: Semi-autoregressive sampling — the causal block factorization

Now draft the whole block. The semi-AR distribution factorizes causally:
$$P(X\mid x_0)=\prod_{k=1}^{\gamma} p_k(x_k\mid x_0,x_{<k}),\qquad p_k=\mathrm{softmax}\big(U_k + B(x_{k-1},\cdot)\big).$$
Sample left-to-right: at each position, add the Markov bias **for the token you just sampled** to that position's base logit, softmax, sample, feed it forward. The very first position conditions on the anchor $x_0$.

**Insight (idea-novel)**: a *parallel* drafter scores every position from the context alone, so it marginalizes over all possible predecessors and produces incoherent suffix combinations ("multi-modal collision"). Feeding the **sampled** previous token back in collapses that marginalization onto one consistent path — the whole point of the sequential stage.
**Self-explain (one line, not checked)**: why does conditioning on the sampled prefix mitigate the "of course" → "of problem" failure mode?

**Predict before you code**: will greedy-decoding *with* the Markov bias produce a different token block than `base_logits.argmax(-1)` (the independent/parallel baseline)? At how many positions, roughly?

In [ ]:
predicted_n_differ = ...   # at how many of the GAMMA positions will the semi-AR block differ from independent argmax(U_k)?

@torch.no_grad()
def draft_block(base_logits: Float[Tensor, "gamma V"], anchor_token: int,
                W1: nn.Embedding, W2: nn.Linear, greedy: bool = True):
    """Sample a γ-token block under p_k = softmax(U_k + B(x_{k-1}, ·)), left to right.
    Returns (tokens: Int[gamma], probs: Float[gamma, V]). Each position must condition
    on the token SAMPLED at the previous position (anchor_token for position 0)."""
    gamma, vocab = base_logits.shape
    # Implement the sequential loop from memory.
    pass

In [ ]:
# --- Part 2 Validation ---
tokens, probs = draft_block(base_logits, anchor_token, W1, W2, greedy=True)
independent = base_logits.argmax(dim=-1)                 # the parallel baseline (no transition bias)
n_differ = int((tokens != independent).sum())

print(f"  You predicted {predicted_n_differ} differing positions; actual {n_differ} of {GAMMA}")
assert tokens.shape == (GAMMA,) and probs.shape == (GAMMA, V), "shape"
print(f"  tokens {tuple(tokens.shape)}  probs {tuple(probs.shape)} -- correct")
assert torch.allclose(probs.sum(-1), torch.ones(GAMMA), atol=1e-5), "each p_k must be a valid distribution (sum to 1)"
print(f"  All γ rows of probs sum to 1 -- correct")

# Reference: reconstruct each step's distribution from the student's OWN sampled prefix.
# probs[k] must equal softmax(U_k + B(x_{k-1})), with x_{-1}=anchor. This pins the causal recurrence.
prev_ids = torch.cat([torch.tensor([anchor_token]), tokens[:-1]])   # x_0 .. x_{γ-2}
ref_probs = torch.softmax(base_logits + markov_bias(prev_ids, W1, W2), dim=-1)
max_diff = (probs - ref_probs).abs().max()
try:
    assert torch.allclose(probs, ref_probs, atol=1e-5), f"Max diff {max_diff:.2e}"
except AssertionError:
    print("  Your probs[1] do not match softmax(U_1 + B(token_0)) -- you may be conditioning on the")
    print("  base-logit argmax or the anchor instead of the token you actually sampled at the prev step.")
    raise
print(f"  Causal recurrence p_k = softmax(U_k + B(x_(k-1))) -- correct (max diff {max_diff:.2e})")
assert n_differ > 0, "the transition bias did nothing -- semi-AR must diverge from the independent baseline somewhere"
print(f"  Diverges from independent baseline at {n_differ} positions -- the sequential head matters\nPart 2 complete.")

## Part 3: Confidence head — predicting per-position acceptance

To verify *smartly* we first need to predict, per position, how likely a draft token is to survive. DSpark's confidence head is a single linear projection + sigmoid over the concatenation of the **backbone hidden** $h_k$ and the **Markov embedding** $W_1[x_{k-1}]$ of the previous token:
$$c_k = \sigma\!\big(w^\top[\,h_k\,;\,W_1[x_{k-1}]\,]\big)\in(0,1).$$
Note it reuses the *same* $W_1$ as Part 1 — that shared embedding is what ties the draft head and the verifier together.

**Predict before you code**: what shape is the input to `conf_proj`, and what shape comes out after the sigmoid + squeeze?

In [ ]:
predicted_concat_dim = ...   # last-dim size of the concatenated [h_k ; W1[x_{k-1}]] input to conf_proj

def confidence_scores(h: Float[Tensor, "gamma d"], prev_emb: Float[Tensor, "gamma r"],
                      conf_proj: nn.Linear) -> Float[Tensor, "gamma"]:
    """c_k = σ( w^T [h_k ; W_1[x_{k-1}]] ).  Returns a probability per position."""
    # Implement from memory.
    pass

In [ ]:
# --- Part 3 Validation ---
torch.manual_seed(42)
prev_ids = torch.cat([torch.tensor([anchor_token]), tokens[:-1]])   # x_0 .. x_{γ-2} from Part 2
prev_emb = W1(prev_ids)
c = confidence_scores(hidden, prev_emb, conf_proj)

print(f"  You predicted concat input dim {predicted_concat_dim}; actual {d + r}")
assert c.shape == (GAMMA,), f"Shape: expected ({GAMMA},), got {tuple(c.shape)}"
print(f"  Shape: {tuple(c.shape)} -- correct")
print(f"  confidences: {[f'{v:.3f}' for v in c.tolist()]}")
assert (c > 0).all() and (c < 1).all(), "every confidence must be a probability in (0,1) -- did you forget the sigmoid?"
print(f"  All in (0,1) -- correct\nPart 3 complete.")

## Part 4: Analytical acceptance label — what the confidence head learns

The confidence head is supervised by the *true* per-step acceptance rate, which has a closed form: the speculative-sampling rule accepts a draw $x\sim p^d$ with probability $\min(1, p^t(x)/p^d(x))$, so the expected acceptance is
$$c_k^{*} = 1 - \tfrac{1}{2}\lVert p_k^{d}-p_k^{t}\rVert_1.$$

**Insight (idea-novel)**: that quantity equals $\sum_v \min(p^d(v), p^t(v))$ — the **overlapping probability mass** of the draft and target distributions. Total-variation distance and acceptance rate are two faces of the same number.
**Self-explain (one line, not checked)**: why does $\sum_v \min(p^d,p^t)$ equal the accept probability of the $\min(1,\,p^t/p^d)$ rule?

**Predict before you code**: what is $c^{*}$ when $p^d = p^t$? When the two distributions have disjoint support?

In [ ]:
predicted_cstar_equal    = ...   # c* when p_d == p_t
predicted_cstar_disjoint = ...   # c* when p_d and p_t have disjoint support

def acceptance_label(p_draft: Float[Tensor, "gamma V"], p_target: Float[Tensor, "gamma V"]) -> Float[Tensor, "gamma"]:
    """Analytical per-step acceptance rate c_k* = 1 - 1/2 * ||p_draft - p_target||_1."""
    # Implement from memory.
    pass

In [ ]:
# --- Part 4 Validation ---
torch.manual_seed(42)
p_draft  = torch.softmax(torch.randn(GAMMA, V), dim=-1)
p_target = torch.softmax(torch.randn(GAMMA, V), dim=-1)
cstar = acceptance_label(p_draft, p_target)

# Prediction reveal: sanity-check the two limiting cases the student committed to.
same = acceptance_label(p_draft, p_draft)
disjoint = acceptance_label(
    torch.tensor([[1.0, 0.0]]), torch.tensor([[0.0, 1.0]]))
print(f"  You predicted c*(p_d==p_t)={predicted_cstar_equal}; actual {same.mean():.3f}")
print(f"  You predicted c*(disjoint)={predicted_cstar_disjoint}; actual {disjoint.item():.3f}")

assert cstar.shape == (GAMMA,), f"Shape {tuple(cstar.shape)}"
print(f"  Shape: {tuple(cstar.shape)}   values: {[f'{v:.3f}' for v in cstar.tolist()]}")
assert (cstar >= 0).all() and (cstar <= 1).all(), "an acceptance rate must lie in [0,1]"

# Reference: 1 - 1/2||p_d-p_t||_1 must equal the overlapping mass sum_v min(p_d, p_t).
ref = torch.minimum(p_draft, p_target).sum(dim=-1)
max_diff = (cstar - ref).abs().max()
try:
    assert torch.allclose(cstar, ref, atol=1e-6), f"Max diff {max_diff:.2e}"
except AssertionError:
    print(f"  YOUR:     {cstar[:4].tolist()}")
    print(f"  EXPECTED (sum of min overlap): {ref[:4].tolist()}")
    raise
print(f"  Matches sum_v min(p_d, p_t) -- TV distance and acceptance are the same number (max diff {max_diff:.2e})")
print("Part 4 complete.")

## Part 5: Hardware-aware prefix scheduler (Algorithm 1)

Producing a long draft block is free; *verifying* all of it is not — under load, every extra verified token steals batch capacity from other requests. So DSpark chooses a per-request verification length $\ell_r$ to maximize system throughput $\Theta=\tau\cdot\mathrm{SPS}(B)$, where the batch size is $B=\sum_r(1+\ell_r)$ and the expected accepts are $\tau=\sum_r\big(1+\sum_{j\le\ell_r} a_{r,j}\big)$, with prefix survival $a_{r,j}=\prod_{i\le j} c_{r,i}$.

The trick (Algorithm 1): build the global pool of candidate tokens $(r,j)$, **sort descending by survival** $a_{r,j}$, then greedily admit them one at a time — each admission is $\ell_r{\leftarrow}j$, $B{+}{=}1$, $\tau{+}{=}a_{r,j}$ — tracking $\Theta$, and **stop the moment $\Theta$ first decreases**.

**Insight (idea-novel)**: survival is *multiplicative* ($a_{r,j}\le a_{r,j-1}$), so the marginal gain of extending a prefix is non-increasing — that monotonicity is exactly what lets a single global greedy sort find the optimum. The early-stop isn't just efficiency: breaking *before* admitting a token keeps the decision from depending on a token you haven't committed to, which is what makes the scheme **lossless** (the non-anticipating property).
**Self-explain (one line, not checked)**: why would continuing past the first $\Theta$ decrease risk leaking a future token into the admission decision?

**Predict before you code**: as you admit more tokens, $\tau$ rises but $\mathrm{SPS}(B)$ falls — so $\Theta$ as a function of #admitted is shaped like ______ ?

<details><summary>Hint 1: survival</summary>Each request's prefix-survival vector is a <em>cumulative product</em> of its confidences along the block dim. <code>torch.cumsum</code> has a multiplicative sibling — reach for it.</details>

In [ ]:
predicted_theta_shape = "..."   # one word: how Θ(#admitted) is shaped (e.g. "monotone-increasing", "unimodal", "flat")

def schedule_prefixes(conf: Float[Tensor, "R gamma"], sps) -> Int[Tensor, "R"]:
    """Algorithm 1. Return per-request verification lengths ℓ_r in [0, gamma] that maximize
    Θ = τ · SPS(B), with B = R + #admitted, τ = R + sum of admitted survivals.
    Greedily admit (r,j) tokens in descending survival order; early-stop at the first Θ drop."""
    R, gamma = conf.shape
    # 1. survival[r, j] = prefix-survival a_{r,j} (cumulative product of conf along the block dim)
    # 2. pool all (r, j) candidates, sort DESCENDING by survival
    # 3. greedily admit candidates one at a time, tracking B, tau and Θ=τ·SPS(B) per the formulas above
    #    (derive the ℓ=0 baseline for B and tau yourself); keep the best assignment seen and
    #    STOP at the first Θ decrease (the losslessness rule).
    pass

In [ ]:
# --- Part 5 Validation ---
torch.manual_seed(42)
R = 3
conf_batch = torch.rand(R, GAMMA) * 0.4 + 0.55          # confidences in (0.55, 0.95)
lengths = schedule_prefixes(conf_batch, sps)

print(f"  You predicted Θ(#admitted) is '{predicted_theta_shape}'")
assert lengths.shape == (R,), f"Shape {tuple(lengths.shape)}"
assert (lengths >= 0).all() and (lengths <= GAMMA).all(), "each ℓ_r must lie in [0, γ]"
print(f"  scheduled lengths: {lengths.tolist()}  (γ={GAMMA}) -- valid range")

# helper: throughput at a given length assignment
survival_full = torch.cumprod(conf_batch, dim=1)
def throughput(L):
    B = R + sum(L)
    tau = float(R) + sum(survival_full[i, :L[i]].sum().item() for i in range(R))
    return tau * sps(B)

theta_student = throughput(lengths.tolist())
theta_zero    = throughput([0] * R)
assert theta_student >= theta_zero - 1e-9, "scheduling must not do worse than verifying nothing extra"

# Reference: brute-force the optimum over all (γ+1)^R length assignments.
best_theta, best_L = -1.0, None
for L in itertools.product(range(GAMMA + 1), repeat=R):
    t = throughput(L)
    if t > best_theta:
        best_theta, best_L = t, L
print(f"  greedy Θ = {theta_student:.5f}   brute-force optimum Θ = {best_theta:.5f}   (brute lengths {list(best_L)})")
max_diff = abs(theta_student - best_theta)
try:
    assert abs(theta_student - best_theta) < 1e-6, f"Θ gap {max_diff:.2e} -- greedy is not optimal here"
except AssertionError:
    print(f"  Your lengths {lengths.tolist()} give Θ={theta_student:.5f}; optimum {list(best_L)} gives {best_theta:.5f}")
    print("  Check: are you sorting by survival DESCENDING, and breaking at the FIRST Θ decrease?")
    raise
print(f"  Greedy admission matches the global optimum (Θ gap {max_diff:.2e})\nPart 5 complete.")

## Part 6: Position-weighted combined loss (with a cross-entropy recall)

Training jointly fits the draft distribution and the confidence head. Three terms, each **position-weighted** by $w_k=\exp(-(k-1)/\gamma)$ (earlier positions matter more under prefix verification):
$$\mathcal{L}_{ce}=-\sum_k w_k\log p_k^d(x_k^{*}),\quad \mathcal{L}_{tv}=\sum_k w_k\lVert p_k^d-p_k^t\rVert_1,\quad \mathcal{L}_{conf}=-\sum_k w_k\big[c_k^{*}\log c_k+(1-c_k^{*})\log(1-c_k)\big]$$
$$\mathcal{L}=\alpha_{ce}\mathcal{L}_{ce}+\alpha_{tv}\mathcal{L}_{tv}+\alpha_{conf}\mathcal{L}_{conf},\qquad (\alpha_{ce},\alpha_{tv},\alpha_{conf})=(0.1,\,0.9,\,1.0).$$

The $\mathcal{L}_{ce}$ term is a **recall**: it is the position-weighted negative-log-likelihood of the true token under $p^d$ — you have built cross-entropy / NLL before (KL-Divergence, the sampling problems). Reproduce it from memory; only the position weighting is new here. The other two reuse Part 4's TV quantity and a plain binary cross-entropy.

**Predict before you code**: with weight $w_k=\exp(-(k-1)/\gamma)$, what is $w_1$ (the first position)? Is the weighting larger early or late in the block?

In [ ]:
predicted_w_first = ...   # value of w_k at k=1 (the first block position)

def dspark_loss(p_draft: Float[Tensor, "gamma V"], p_target: Float[Tensor, "gamma V"],
                target_ids: Int[Tensor, "gamma"], c: Float[Tensor, "gamma"], c_star: Float[Tensor, "gamma"],
                gamma: int, alpha_ce=0.1, alpha_tv=0.9, alpha_conf=1.0) -> Tensor:
    """Position-weighted L = α_ce·L_ce + α_tv·L_tv + α_conf·L_conf  (see formulas above)."""
    # w_k = exp(-(k-1)/gamma) for k = 1..gamma  (0-indexed: exp(-i/gamma))
    # L_ce   = -sum_k w_k * log p_draft[k, target_ids[k]]          (RECALL: weighted NLL)
    # L_tv   =  sum_k w_k * ||p_draft[k] - p_target[k]||_1
    # L_conf = -sum_k w_k * [ c_star*log(c) + (1-c_star)*log(1-c) ]
    pass

In [ ]:
# --- Part 6 Validation ---
torch.manual_seed(42)
p_draft  = torch.softmax(torch.randn(GAMMA, V), dim=-1)
p_target = torch.softmax(torch.randn(GAMMA, V), dim=-1)
target_ids = torch.randint(0, V, (GAMMA,))
c      = torch.rand(GAMMA) * 0.9 + 0.05
c_star = torch.rand(GAMMA) * 0.9 + 0.05
loss = dspark_loss(p_draft, p_target, target_ids, c, c_star, GAMMA)

w = torch.exp(-torch.arange(GAMMA).float() / GAMMA)
print(f"  You predicted w_1 = {predicted_w_first}; actual {w[0].item():.3f}  (weights: {[f'{x:.2f}' for x in w.tolist()]})")
assert loss.ndim == 0, "loss must be a scalar"
print(f"  loss = {loss.item():.5f}  (scalar) -- correct")

# Reference: rebuild each term independently (weighted NLL, weighted L1, weighted BCE) and recombine.
ref_ce = (w * F.nll_loss(torch.log(p_draft), target_ids, reduction="none")).sum()
ref_tv = (w * (p_draft - p_target).abs().sum(-1)).sum()
ref_conf = (w * F.binary_cross_entropy(c, c_star, reduction="none")).sum()
ref = 0.1 * ref_ce + 0.9 * ref_tv + 1.0 * ref_conf
max_diff = (loss - ref).abs().item()
print(f"  term breakdown: ce={ref_ce.item():.4f}  tv={ref_tv.item():.4f}  conf={ref_conf.item():.4f}")
try:
    assert torch.allclose(loss, ref, atol=1e-5), f"diff {max_diff:.2e}"
except AssertionError:
    print(f"  YOUR loss {loss.item():.6f}  vs  REFERENCE {ref.item():.6f}")
    print("  Check the position weights w_k=exp(-(k-1)/γ) and the three α coefficients (0.1, 0.9, 1.0).")
    raise
print(f"  Reference match (independent term-by-term) -- correct (diff {max_diff:.2e})\nPart 6 complete.")

## Putting it together: one DSpark draft → confidence → schedule round

This wires Parts 1-5 into a single decoding round using **your** functions — no new code to write,
just run it and read the trace. The backbone drafts a block (semi-AR), the confidence head scores each
position, and the scheduler decides how much of a batch of look-alike requests to actually verify.

In [ ]:
# 1. Draft a γ-token block with the semi-AR Markov head (Parts 1-2)
tokens, probs = draft_block(base_logits, anchor_token, W1, W2, greedy=True)

# 2. Score per-position confidence (Part 3), using the shared Markov embedding of the previous token
prev_ids = torch.cat([torch.tensor([anchor_token]), tokens[:-1]])
conf_1req = confidence_scores(hidden, W1(prev_ids), conf_proj)

# 3. Schedule verification length across a batch of 3 look-alike requests under load (Parts 4-5 machinery)
conf_batch = (conf_1req.unsqueeze(0) + 0.05 * torch.randn(3, GAMMA)).clamp(1e-3, 0.999)
lengths = schedule_prefixes(conf_batch, sps)

print("drafted block tokens     :", tokens.tolist())
print("per-position confidence  :", [f"{v:.2f}" for v in conf_1req.tolist()])
print("scheduled verify lengths :", lengths.tolist(), f"(out of γ={GAMMA} per request)")
print("\nUnder this load the scheduler verifies only the high-survival prefix of each request,")
print("spending target-model batch capacity where the expected accepted length is highest.")

## Session Debrief

Write your answers into the code cell below (typing is overt retrieval — far stronger than answering
"in your head"). Don't scroll up.
1. What does the sequential (Markov) head add to the parallel backbone's base logits, and what factorization does that induce over the block?
2. The analytical acceptance label $c_k^{*}=1-\tfrac12\lVert p^d-p^t\rVert_1$ equals what other simple quantity over the two distributions?
3. In the prefix scheduler, why does sorting candidates by survival $a_{r,j}$ and stopping at the first $\Theta$ drop yield the optimum — and what does the early-stop buy you besides speed?

**Check yourself**: your worked solution is in `_solutions/dspark.ipynb` — open it (and the paper) to grade
your answers. Opening it ends the retrieval rep, so answer first.

**Challenge**: Close this notebook, open a blank one, and rewrite **Part 5 (the prefix scheduler)** from
scratch without looking back — it's the part with the most moving pieces.

In [ ]:
debrief = """
1.
2.
3.
"""  # type your recall here before checking _solutions/

In [ ]:
# --- Session log: fill the `___` then run (appends one line to .practice-log.jsonl) ---
import json, datetime, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / ".git").exists() or (p / ".spaced-repetition.json").exists())
record = {
    "problem": "papers/dspark",
    "date": datetime.date.today().isoformat(),
    "budget_min": 60,
    "actual_min": ___,                                  # how long it really took
    "parts": [                                          # n + tier pre-filled; set outcome
        {"n": 1, "tier": 3, "outcome": "___"},          # unaided | hint | solution | failed
        {"n": 2, "tier": 3, "outcome": "___"},
        {"n": 3, "tier": 3, "outcome": "___"},
        {"n": 4, "tier": 3, "outcome": "___"},
        {"n": 5, "tier": 2, "outcome": "___"},
        {"n": 6, "tier": 3, "outcome": "___"},
    ],
    "difficulty": ___,                                  # 1 (trivial) .. 5 (over my head)
    "stuck": "___",                                     # one phrase: where you got stuck
}
with open(_root / ".practice-log.jsonl", "a") as f:
    f.write(json.dumps(record) + "\n")
print("logged ->", _root / ".practice-log.jsonl")

## Extension (Optional)
- **RNN head**: replace the Markov head with the paper's gated RNN head — it carries a recurrent state $s_k$ so position $k$ can see the *whole* prefix, not just $x_{k-1}$. Implement $s_k=\sigma(W_g z_k)\odot s_{k-1}+(1-\sigma(W_g z_k))\odot\tanh(W_c z_k)$ with $z_k=[s_{k-1};W_1[x_{k-1}];h_k]$, and bias $B=W_2^\top\tanh(W_o z_k)$.
- **Break it on purpose**: in `draft_block`, condition the bias on `independent.argmax` (the *base-logit* argmax) instead of the token you sampled. Re-run Part 2's validation and explain which assert fires and why.
- **Sequential Temperature Scaling**: the raw confidences are overconfident, so DSpark calibrates the cumulative product $\prod_{i\le k} c_i$ left-to-right with a per-position temperature chosen by 1-D grid search to minimize ECE. Implement the grid search for one position.

## Anki Cards

**Card 1**
Front: A parallel speculative drafter proposes "of course" then "of problem" — coherent first token, garbage suffix. What mechanism fixes this without going fully autoregressive?
Back: Semi-AR transition bias (Markov head)

**Card 2**
Front: You need the per-step acceptance rate of speculative sampling but only have the draft and target distributions. What do you compute?
Back: 1 − ½‖p_d − p_t‖₁ (= Σ min)

**Card 3**
Front: Your confidence-scheduled verifier is leaking future tokens into its admission decision and breaking losslessness. What rule restores the non-anticipating property?
Back: Early-stop at first Θ drop